In [1]:

!CONDA_BASE="$HOME/miniconda3"
!source "$CONDA_BASE/etc/profile.d/conda.sh"

!conda env list
!which conda
import sys
print("Python path:", sys.executable)
print("Python version:", sys.version)
# Check for GPU with PyTorch
import torch
print(torch.cuda.is_available())

# Check for GPU with TensorFlow
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

import tensorflow as tf
print(tf.__version__)  # Should show 2.15.x
print("GPU Available:", tf.config.list_physical_devices('GPU'))

/bin/bash: line 1: /etc/profile.d/conda.sh: No such file or directory

# conda environments:
#
nnunet_env             /home/rbielski/.conda/envs/nnunet_env
pycharm_env            /home/rbielski/.conda/envs/pycharm_env
stroke_sota            /home/rbielski/.conda/envs/stroke_sota
tf215_env              /home/rbielski/.conda/envs/tf215_env
tf_2_15                /home/rbielski/.conda/envs/tf_2_15
base                   /home/rbielski/miniconda3
geo_env                /home/rbielski/miniconda3/envs/geo_env
stroke_env           * /home/rbielski/miniconda3/envs/stroke_env
tf215_env_recreated    /home/rbielski/miniconda3/envs/tf215_env_recreated

/home/rbielski/miniconda3/condabin/conda
Python path: /home/rbielski/miniconda3/envs/stroke_env/bin/python
Python version: 3.10.14 | packaged by conda-forge | (main, Mar 20 2024, 12:45:18) [GCC 12.3.0]
True


2025-09-09 15:56:16.566839: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-09 15:56:16.566872: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-09 15:56:16.567916: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-09 15:56:16.572842: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-09 15:56:17.187498: W tensorflow/compiler/tf2

[]
2.15.0
GPU Available: []


2025-09-09 15:56:17.758699: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-09-09 15:56:17.760348: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-09-09 15:56:17.762357: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required l

In [9]:
# 📊 ROBUST DATASET PREPARATION with CROPPED_COMBINED Atlas Data (self-contained)
import numpy as np
from pathlib import Path

print("🔍 Setting up dataset loading...")

# Minimal local config (no external training module needed)
class TrainingConfig:
    def __init__(self):
        self.DATA_DIR = Path("/home/rbielski/Atlas_2/Training/Cropped_128_Combined")
        self.INPUT_SHAPE = (128, 128, 128, 1)

# Create or ensure config exists
if 'config' not in globals() or config is None:
    config = TrainingConfig()
    print("📝 Created new config object")

# Ensure DATA_DIR is a Path object
if not hasattr(config, 'DATA_DIR') or config.DATA_DIR is None:
    config.DATA_DIR = Path("/home/rbielski/Atlas_2/Training/Cropped_128_Combined")
else:
    config.DATA_DIR = Path(config.DATA_DIR)

# Helper: detect flat combined dataset (no Images/Masks subfolders)
def has_flat_combined_dir(p: Path) -> bool:
    if not p.exists():
        return False
    imgs = list(p.glob("*nii_T1w_cropped128.nii.gz"))
    msks = list(p.glob("*nii_mask_cropped128.nii.gz"))
    return len(imgs) > 0 and len(msks) > 0

CROPPED_DEFAULT = Path("/home/rbielski/Atlas_2/Training/Cropped_128_Combined")

# Find a valid data directory (PRIORITIZE flat Cropped_128_Combined)
fallback_paths = [
    CROPPED_DEFAULT,
    Path("../Atlas_2/Training/Cropped_128_Combined"),
    Path("../../Atlas_2/Training/Cropped_128_Combined"),
    Path("./Atlas_2/Training/Cropped_128_Combined"),
    config.DATA_DIR,  # last: respect prior config if above didn't match flat
]

data_dir_found = None
layout = None  # 'flat' or 'split'
for candidate_path in fallback_paths:
    if not candidate_path.exists():
        print(f"✗ Not found: {candidate_path}")
        continue
    if has_flat_combined_dir(candidate_path):
        data_dir_found = candidate_path
        layout = 'flat'
        print(f"✅ Using flat combined dataset at: {data_dir_found}")
        break
    if (candidate_path / "Images").exists() and (candidate_path / "Masks").exists():
        data_dir_found = candidate_path
        layout = 'split'
        print(f"✅ Using split dataset (Images/Masks) at: {data_dir_found}")
        break
    print(f"✗ Not matching known layouts: {candidate_path}")

if data_dir_found is None:
    print("❌ No valid Atlas data directory found!")
    print("Expected either flat files '*T1w_cropped128.nii.gz' + '*mask_cropped128.nii.gz' or split Images/ + Masks/ folders")
    pairs, lesion_presence = [], []
else:
    # Update config to the selected directory
    config.DATA_DIR = data_dir_found

    print(f"📚 Loading dataset manually (layout={layout})...")
    if layout == 'split':
        images_dir = data_dir_found / "Images"
        masks_dir = data_dir_found / "Masks"
        image_patterns = ['*nii_T1w_cropped128.nii.gz', '*_t1.nii.gz', '*T1w.nii.gz', '*t1w.nii.gz']
        mask_patterns = ['*nii_mask_cropped128.nii.gz', '*_lesion.nii.gz', '*_label-L*.nii.gz']
        images = []
        for pattern in image_patterns:
            found_images = list(images_dir.glob(pattern))
            if found_images:
                images.extend(found_images)
                break
        masks = []
        for pattern in mask_patterns:
            found_masks = list(masks_dir.glob(pattern))
            if found_masks:
                masks.extend(found_masks)
                break
    else:  # flat combined
        images = sorted(data_dir_found.glob("*nii_T1w_cropped128.nii.gz"))
        masks = sorted(data_dir_found.glob("*nii_mask_cropped128.nii.gz"))

    # Build key maps for robust pairing
    def basekey_from_name(name: str) -> str:
        if name.endswith("nii_T1w_cropped128.nii.gz"):
            return name[:-len("nii_T1w_cropped128.nii.gz")]
        if name.endswith("nii_mask_cropped128.nii.gz"):
            return name[:-len("nii_mask_cropped128.nii.gz")]
        # Generic fallback: before first .nii
        return name.split(".nii")[0]

    image_map = {basekey_from_name(p.name): p for p in images}
    mask_map = {basekey_from_name(p.name): p for p in masks}

    # Intersect keys to form pairs
    common_keys = sorted(set(image_map.keys()) & set(mask_map.keys()))
    pairs = [(image_map[k], mask_map[k]) for k in common_keys]
    lesion_presence = [1] * len(pairs)

    # Report unmatched files (useful for debugging naming mismatches)
    missing_masks = sorted(k for k in image_map.keys() if k not in mask_map)
    missing_images = sorted(k for k in mask_map.keys() if k not in image_map)
    if missing_masks:
        print(f"ℹ️ Images without masks: {len(missing_masks)} (e.g., {missing_masks[:3]})")
    if missing_images:
        print(f"ℹ️ Masks without images: {len(missing_images)} (e.g., {missing_images[:3]})")

    print(f"✅ Manually paired {len(pairs)} image-mask pairs")
    if len(pairs) > 0:
        from pathlib import Path as _P
        print("   Examples:")
        for i, (img, msk) in enumerate(pairs[:3]):
            print(f"   {i+1:>2}. {_P(img).name}  <->  {_P(msk).name}")

# Summary
if len(pairs) > 0:
    print(f"\n📊 DATASET SUMMARY:")
    print(f"   Total pairs: {len(pairs)}")
    if len(lesion_presence) > 0:
        print(f"   Lesion presence: {np.mean(lesion_presence)*100:.1f}% of samples")
    print(f"   Data directory: {config.DATA_DIR}")
    try:
        from pathlib import Path as _P
        print(f"   Example pair: {_P(pairs[0][0]).name} <-> {_P(pairs[0][1]).name}")
    except Exception:
        pass
else:
    print(f"\n❌ No dataset pairs loaded!")

print(f"\nDataset ready: {len(pairs)} pairs available for testing")

🔍 Setting up dataset loading...
✅ Using flat combined dataset at: /home/rbielski/Atlas_2/Training/Cropped_128_Combined
📚 Loading dataset manually (layout=flat)...
✅ Manually paired 655 image-mask pairs
   Examples:
    1. sub-r001s001_ses-1_space-MNI152NLin2009aSym.nii_T1w_cropped128.nii.gz  <->  sub-r001s001_ses-1_space-MNI152NLin2009aSym.nii_mask_cropped128.nii.gz
    2. sub-r001s002_ses-1_space-MNI152NLin2009aSym.nii_T1w_cropped128.nii.gz  <->  sub-r001s002_ses-1_space-MNI152NLin2009aSym.nii_mask_cropped128.nii.gz
    3. sub-r001s003_ses-1_space-MNI152NLin2009aSym.nii_T1w_cropped128.nii.gz  <->  sub-r001s003_ses-1_space-MNI152NLin2009aSym.nii_mask_cropped128.nii.gz

📊 DATASET SUMMARY:
   Total pairs: 655
   Lesion presence: 100.0% of samples
   Data directory: /home/rbielski/Atlas_2/Training/Cropped_128_Combined
   Example pair: sub-r001s001_ses-1_space-MNI152NLin2009aSym.nii_T1w_cropped128.nii.gz <-> sub-r001s001_ses-1_space-MNI152NLin2009aSym.nii_mask_cropped128.nii.gz

Dataset re